<a href="https://colab.research.google.com/github/Amnaikram1/Amnaikram1/blob/master/Daruliftaa_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
df = pd.read_excel('/content/Questions.xlsx')

In [11]:
!pip install sentence_transformers
!pip install faiss-cpu
!pip install html
!pip install bs4

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Encode Answer column
embeddings = model.encode(
    df["Answer"].tolist(),
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

df["embedding"] = embeddings.tolist()  # convert to list for PostgreSQL


In [12]:
import pandas as pd
import numpy as np
import faiss
import html
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import time

# -------------------- TEXT CLEANER --------------------
class TextCleaner:
    """
    Cleans HTML, CSS, and encoded text
    """
    @staticmethod
    def clean(text: str) -> str:
        if pd.isna(text):
            return text
        text = html.unescape(str(text))
        soup = BeautifulSoup(text, "html.parser")
        for tag in soup(["script", "style"]):
            tag.decompose()
        return soup.get_text(separator=" ", strip=True)

# -------------------- FAWA SEMANTIC SEARCH --------------------
class FatwaSemanticSearch:
    """
    Semantic search engine using FAISS + SentenceTransformers
    Precomputed embeddings are loaded from .npy
    """
    def __init__(self, df: pd.DataFrame, embeddings_path: str, model_name: str = "paraphrase-multilingual-MiniLM-L12-v2"):
        # -------------------- CLEAN DATA --------------------
        df = df.dropna(subset=["Answer"])
        df = df[df["Answer"].astype(str).str.strip() != ""]
        df = df.reset_index(drop=True)

        df["Statement"] = df["Statement"].apply(TextCleaner.clean)
        df["Answer"] = df["Answer"].apply(TextCleaner.clean)
        self.df = df

        # -------------------- LOAD EMBEDDINGS --------------------
        self.embeddings = np.load(embeddings_path).astype("float32")
        assert self.embeddings.shape[0] == len(self.df)

        # -------------------- LOAD MODEL --------------------
        self.model = SentenceTransformer(model_name)

        # -------------------- BUILD FAISS INDEX --------------------
        self.index = self._build_index()
        print(f"FAISS index built with {self.index.ntotal} vectors, dimension {self.index.d}")

    def _build_index(self):
        dim = self.embeddings.shape[1]
        index = faiss.IndexFlatIP(dim)  # inner product = cosine similarity if normalized
        index.add(self.embeddings)
        return index

    def search(self, query: str, top_k: int = 5):
        query_vector = self.model.encode(query, normalize_embeddings=True).astype("float32").reshape(1, -1)

        if query_vector.shape[1] != self.index.d:
            raise ValueError(f"Query dimension ({query_vector.shape[1]}) does not match index dimension ({self.index.d})")

        scores, indices = self.index.search(query_vector, top_k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            results.append({
                "score": float(score),
                "category": self.df.iloc[idx]["Category"],
                "answer": self.df.iloc[idx]["Answer"],
                "Statement": self.df.iloc[idx]["Statement"],
                "Title": self.df.iloc[idx]["Title"]
            })
        return results

# -------------------- USAGE --------------------

search_engine = FatwaSemanticSearch(df=df, embeddings_path="/content/fatwa_embeddings.npy")

query = """What is the ruling of the length of the beard according to the Ḥanafīs ? I hear from the Desi subcontinent that fist length is wājib . However, the Syrian/Arab and Turkish Ḥanafis do not seem to agree with this. Additionally, there are some prominent Arab Ḥanafī ʿ ulamāʾ who have written papers saying that fist length is not wājib for Ḥanafīs , such as Dr. Salah. Is there a difference within the madhhab regarding this?"""

results = search_engine.search(query, top_k=10)

for r in results:
    print(f"Title: {r['Title']}")
    print(f"Fatwa: {r['answer']}\n")


AttributeError: partially initialized module 'torch' has no attribute 'autograd' (most likely due to a circular import)